<a href="https://colab.research.google.com/github/jinbumlee95/Python_CLI_Board_Practice/blob/main/Python_CLI_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from IPython.display import clear_output
import pandas as pd
import traceback
from datetime import date, datetime

running = True

# pandas 를 임시 데이터 베이스로 사용하도록 처리.
try:
    article_list = pd.read_csv("article_list.csv")
except FileNotFoundError:
    article_list = pd.DataFrame(columns=["No", "title", "desc","createdby","createdon","updatedby","lastupdated"])
    article_list.to_csv('article_list.csv',index=False)

try:
    member_list = pd.read_csv("member_list.csv",dtype="str")
except FileNotFoundError:
    member_list = pd.DataFrame(columns=["memberid", "password", "membername"])
    member_list.to_csv('member_list.csv',index=False)


def write_article() :
    global article_list
    global session_id
    # 1. 새로운 글 번호 계산 (기존 데이터가 있으면 max + 1, 없으면 1)
    if article_list.empty:
        new_no = 1
    else:
        new_no = article_list['No'].max() + 1

    title = input("제목 : ")
    print()
    desc = input("내용 : " )
    print()
    article_list.loc[len(article_list)] = [new_no, title, desc,session_id,date.today(),None,None]
    article_list.to_csv('article_list.csv', index=False) # 파일에 저장
    print("{}번 글이 생성되었습니다.".format(new_no))

def list_article():
    global article_list
    global member_list
    if article_list.empty :
        print('게시글이 존재하지 않습니다.')
    else :
        # 작성자 명을 가져오기 위해서 merge
        list_table = pd.merge(
            article_list,
            member_list,
            left_on='createdby',
            right_on='memberid',
            how='left'
        )



        print(f"{'No':<5} | {'title':<30} | {'createdby':<10}")
        print('--------------------------------------------------')
        for idx, board in list_table.loc[::-1].iterrows() :
            #print(board)
            print(f"{board['No']:<5} | {board['title']:<30} | {board['membername']:<10}")
            print('--------------------------------------------------')

def read_article(n = None) :
    #n = int(n)
    if n == None :
        n = int(input("조회할 게시글 번호 : "))
    else :
        n = int(n)
    board = article_list[article_list['No'] == n]

    if board.empty :
        print("==존재하지 않는 게시글 입니다.==")
    else :
        board = board.iloc[0]
        print("======== 제목 =========")
        print(board['title'])
        print("======== 작성일 =========")
        print(board['createdon'])
        print("======== 내용 =========")
        print(board['desc'])


def delete_article(n = None):
    global article_list

    if n == None :
        n = int(input("삭제할 게시글 번호 : "))
    else :
        n = int(n)

    board = article_list[article_list['No'] == n]
    if board.empty :
        print("==존재하지 않는 게시글 입니다.==")
    elif board[board['createdby'] == session_id].empty :
        print("==본인의 게시글만 삭제 할 수 있습니다.==")
    else :
        # 3. 해당 번호를 제외한 행만 남겨 삭제 처리
        article_list = article_list[article_list['No'] != n].copy()
        # 4. 'No' 컬럼 번호 1부터 재정렬 > 이거는 게시글 순번이 필요하니까 안할거임
        #article_list['No'] = range(1, len(article_list) + 1)
        # 5. CSV 파일 저장
        article_list.to_csv('article_list.csv', index=False)

        print('삭제되었습니다.')

def close_cli() :
     global running
     running = False
     print("=======  CLI 게시판 종료 =========")

def print_help():
    #os.system('cls')

    print("COMMANDS ")
    print('========================================')
    for key in command_dict :
        print(key)
    print('========================================')

def update_article(n = None):
    global session_id
    if n == None :
        n = int(input("수정할 게시글 번호 : "))
    else :
        n = int(n)
    update_board = article_list[article_list['No'] == n]

    if update_board.empty :
        print("==존재하지 않는 게시글 입니다.==")
    elif update_board[update_board['createdby'] == session_id].empty :
        print("==본인의 게시글만 수정 할 수 있습니다.==")
    else :
        update_board = article_list[article_list['No'] == n].iloc[0]

        print('수정할 사항을 선택 해 주세요.')
        print('1 : 제목')
        print('2 : 내용')

        # 1이나 2가 입력될 때까지 계속 입력 받기
        mode = input("선택 ) ")
        while mode not in ['1', '2']:
            print('올바른 번호를 선택해 주세요. (1 : 제목, 2 : 내용)')
            mode = input("선택 ) ")

        if mode == '1':
            title = input('수정할 제목을 입력 해 주세요 : ').strip()
            if title:
                article_list.loc[article_list['No'] == n, 'title'] = title
                article_list.loc[article_list['No'] == n, 'lastupdated'] = date.today()
                article_list.loc[article_list['No'] == n, 'updatedby'] = session_id

        elif mode == '2':
            desc = input('수정할 내용을 입력 해 주세요 : ').strip()
            if desc:
                article_list.loc[article_list['No'] == n, 'desc'] = desc
                article_list.loc[article_list['No'] == n, 'lastupdated'] = date.today()
                article_list.loc[article_list['No'] == n, 'updatedby'] = session_id

        article_list.to_csv('article_list.csv', index=False) # 파일에 저장
        read_article(int(n)) # 수정 후에 수정 한 게시글을 조회 하도록 수정








def login_process(id = None , password = None):
    global session_id
    global membername
    if member_list[(member_list['memberid'] == id) & (member_list['password'] == password)].empty:
        print('존재하지 않는 회원입니다')
    else :
        member = member_list[(member_list['memberid'] == id)].iloc[0]
        session_id = id
        membername = member['membername']
        print(f'{membername}님, 방문을 환영합니다.')


def logout_process() :
    global session_id
    global membername

    print(f'{membername}님, 재방문을 기다리겠습니다.')

    session_id = ''
    membername = ''

def sign_up():
    global member_list

    memberid = input("아이디를 입력해 주세요 : ")

    duplication_check = True
    while duplication_check :
        duplication_check = not member_list[member_list['memberid'] == memberid].empty
        if(duplication_check) :
            memberid = input("중복된 아이디 입니다. 다른 아이디를 입력해주세요. : ")
    print()
    password = input("비밀번호를 입력해 주세요 : " )
    print()
    membername = input("이름을 입력해 주세요 : " )
    print()
    member_list.loc[len(member_list)] = [memberid, password, membername]
    member_list.to_csv('member_list.csv', index=False) # 파일에 저장
    print("{}님 환영합니다. 로그인 후 사용해주세요.".format(membername))


    pass




command_dict ={

    "article write" : write_article,
    "article list" : list_article,
    "article read" : read_article,
    "article delete" : delete_article,
    "article update" : update_article,
    "help" : print_help,
    "clear" : clear_output,
    'login' : login_process,
    'logout' : logout_process,
    'sign' : sign_up,
    "exit" : close_cli
}


session_id = ''
print("=======  CLI 게시판 실행 =========")
while running :

    command = input("명령어 ) ").strip()

    if not command :
        continue

    command_list = command.split()

    # 2단어 명령어(article read 등)와 1단어 명령어(help, exit 등) 구분 처리
    if len(command_list) >= 2 and f"{command_list[0]} {command_list[1]}" in command_dict:
        # 2단 커맨드가 dict 안에 있으면 cmd 를 2단 커맨드로

        cmd = f"{command_list[0]} {command_list[1]}"
        args = command_list[2:]
    else:
        # 아니면 1단 커맨드에 뒤에 나오는건 인자들
        cmd = command_list[0]
        args = command_list[1:]

    if cmd not in  ['login','sign','help','exit','clear'] and session_id == '' :
        print('로그인 후 사용 해주세요.')
        continue
    elif cmd in  ['login','sign'] and session_id != '':
        print('로그아웃 후 사용해주세요.')
        continue
    elif cmd == 'logout' and session_id == '' :
        print('로그인 상태가 아닙니다.')



    if cmd not in command_dict :
        print(f'{command}는 존재하지 않는 명령어 입니다. help 명령어를 통해, 명령어 목록을 확인해주세요.')
    else :
        try :
            if args :
                command_dict[cmd](*args)
            else :
                command_dict[cmd]()
        except:
            # 인자 수를 잘못 넘겨주는 경우에 대해서 예외 처리
            print('잘못된 명령어를 입력하셨습니다.')
            #exc_str = traceback.format_exc()
            #print(exc_str)



=======  CLI 게시판 실행 =========
명령어 ) login member2 1234
JB2님, 방문을 환영합니다.
명령어 ) article read 1
======== 제목 =========
작성되어 주세요 제발
======== 작성일 =========
2026-08-31
======== 내용 =========
제발 업데이트 되어 주세요
명령어 ) article read 2
======== 제목 =========
글 작성 입니다 ㅎㅎㅎ
======== 작성일 =========
2026-08-31
======== 내용 =========
내용이에용 ^^
명령어 ) article read 3
==존재하지 않는 게시글 입니다.==
명령어 ) article read 1
======== 제목 =========
작성되어 주세요 제발
======== 작성일 =========
2026-08-31
======== 내용 =========
제발 업데이트 되어 주세요
명령어 ) article update 1
==본인의 게시글만 수정 할 수 있습니다.==
명령어 ) arcticle delete 1
arcticle delete 1는 존재하지 않는 명령어 입니다. help 명령어를 통해, 명령어 목록을 확인해주세요.
명령어 ) article delete 1
==본인의 게시글만 삭제 할 수 있습니다.==
명령어 ) logout
JB2님, 재방문을 기다리겠습니다.
명령어 ) login member1 1234
JB님, 방문을 환영합니다.
명령어 ) article update 1
수정할 사항을 선택 해 주세요.
1 : 제목
2 : 내용
선택 ) 1
수정할 제목을 입력 해 주세요 : 1번 제목을 주인인 제가 수정 합니다
======== 제목 =========
1번 제목을 주인인 제가 수정 합니다
======== 작성일 =========
2026-08-31
======== 내용 =========
제발 업데이트 되어 주세요
명령어 ) article delete 1
삭제되었습니다.
명령어